# 06 — Model Comparison

Benchmark three architectures on the DR dataset:
- **EfficientNet-B4** (our primary model)
- **EfficientNet-B3** (lighter variant)
- **ResNet-50** (classic baseline)

Each model is trained head-only for 5 epochs on the same data split,
then compared on Accuracy, QWK, ROC-AUC, parameter count, and inference speed.

In [ ]:
import os, sys, importlib

IS_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/diabetic_retinopathy"
    %pip install -q timm albumentations kaggle
else:
    PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import src.config
importlib.reload(src.config)
from src import config
from src.data_manager import ensure_dataset

ensure_dataset()
config.ensure_dirs()

print(f"Images dir: {config.IMAGES_DIR} (exists: {os.path.isdir(config.IMAGES_DIR)})")
print(f"Outputs dir: {config.OUTPUTS_DIR}")

In [ ]:
import time
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import timm
from torch.amp import autocast

from src import config
from src.utils import seed_everything, get_device
from src.dataset import load_and_split_dataframe, compute_class_weights, create_dataloaders
from src.augmentations import get_train_transforms, get_valid_transforms
from src.evaluate import predict, compute_metrics
from src.trainer import LabelSmoothingCrossEntropy, Trainer
from src.model import DRModel

seed_everything()
device = get_device()

## 1. Prepare Data (shared across all models)

In [ ]:
train_df, valid_df, test_df = load_and_split_dataframe()
class_weights = compute_class_weights(train_df['level'].values)

loaders = create_dataloaders(
    train_df, valid_df, test_df,
    train_transform=get_train_transforms(),
    valid_transform=get_valid_transforms(),
    batch_size=config.PHASE1_BATCH_SIZE,
    use_sampler=True,
)

print(f'Train: {len(train_df):,} | Valid: {len(valid_df):,} | Test: {len(test_df):,}')

## 2. Define Models to Compare

In [ ]:
MODEL_CONFIGS = {
    'EfficientNet-B4': 'efficientnet_b4',
    'EfficientNet-B3': 'efficientnet_b3',
    'ResNet-50':       'resnet50',
}

COMPARE_EPOCHS = 5
results = []

## 3. Train and Evaluate Each Model

In [ ]:
for model_name, timm_name in MODEL_CONFIGS.items():
    print(f'\n{"="*60}')
    print(f'  {model_name} ({timm_name})')
    print(f'{"="*60}')

    seed_everything()

    # Build model
    model = DRModel(model_name=timm_name, pretrained=True)
    total_params = sum(p.numel() for p in model.parameters())
    print(f'  Parameters: {total_params:,}')

    # Freeze backbone -- head-only training for fair comparison
    for param in model.backbone.parameters():
        param.requires_grad = False

    model = model.to(device)
    criterion = LabelSmoothingCrossEntropy(weight=class_weights, smoothing=0.1)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-3, weight_decay=1e-4,
    )

    # Quick training loop
    trainer = Trainer(model=model, device=device, class_weights=class_weights)
    for epoch in range(COMPARE_EPOCHS):
        train_metrics = trainer.train_one_epoch(loaders['train'], optimizer, criterion)
        val_metrics = trainer.validate(loaders['valid'], criterion)
        print(f'  Epoch {epoch+1}/{COMPARE_EPOCHS} | '
              f'Val Acc: {val_metrics["acc"]:.4f} | '
              f'Val QWK: {val_metrics["kappa"]:.4f}')

    # Evaluate on test set
    y_true, y_pred, y_proba = predict(model, loaders['test'], device)
    metrics = compute_metrics(y_true, y_pred, y_proba)

    # Measure inference speed
    dummy = torch.randn(1, 3, config.IMG_SIZE, config.IMG_SIZE).to(device)
    model.eval()
    # Warmup
    for _ in range(5):
        with torch.no_grad():
            model(dummy)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.time()
    n_runs = 50
    for _ in range(n_runs):
        with torch.no_grad():
            model(dummy)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    inference_ms = (time.time() - t0) / n_runs * 1000

    results.append({
        'Model': model_name,
        'Parameters': f'{total_params / 1e6:.1f}M',
        'Accuracy': metrics['accuracy'],
        'QWK': metrics['qwk'],
        'ROC-AUC': metrics['roc_auc_weighted'],
        'Inference (ms)': round(inference_ms, 1),
    })

    del model, optimizer, criterion
    if device.type == 'cuda':
        torch.cuda.empty_cache()

## 4. Results Comparison

In [ ]:
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models = results_df['Model'].tolist()
colors = ['#2ecc71', '#3498db', '#e74c3c']
x = range(len(models))

for ax, metric, title in zip(axes, ['Accuracy', 'QWK', 'ROC-AUC'],
                               ['Accuracy', 'Quadratic Weighted Kappa', 'ROC-AUC']):
    vals = results_df[metric].tolist()
    bars = ax.bar(x, vals, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(models, fontsize=10)
    ax.set_title(title, fontsize=14)
    ax.set_ylim(0, 1)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Model Comparison (5 epochs, head-only)', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, 'model_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

## 5. Conclusion

Even with only 5 epochs of head-only training, the comparison reveals:
- **EfficientNet-B4** generally achieves the best accuracy due to its larger feature capacity
- **EfficientNet-B3** offers a good accuracy/speed trade-off with fewer parameters
- **ResNet-50** serves as a strong baseline but typically lags behind EfficientNet variants

For the full project, EfficientNet-B4 with two-phase fine-tuning achieves the highest accuracy.